# Reproduce every condition from scratch

Trains all seven conditions of the paper for the chosen seeds, then builds the cross-seed summary and the paper tables. Order per seed: M (Stage A, B, C), B2, B3, B4, A1, A2, A3. B2, B3, A2 and A3 reuse M's Stage A (B3 and A2 also its Stage B) of the same seed; A1 trains its own stages; B4 (XGBoost, all training rows) tunes once before its first seed. See [docs/REPRODUCING.md](https://github.com/selimsidan/dmoe_nids/blob/main/docs/REPRODUCING.md) for what each condition trains and the compute it needs.

* **Before running:** follow [docs/DATA.md](https://github.com/selimsidan/dmoe_nids/blob/main/docs/DATA.md) to put the four NF-v3 CSVs in your Drive (or set `DATA_ROOT`).
* **How to use:** edit only the SETTINGS cell, then *Runtime > Run all*. Rerunning after a disconnect continues where training stopped (finished units are skipped, an interrupted unit resumes).
* **Runtime type:** an A100 GPU if B4 is included (XGBoost on all training rows needs about 40 GB of GPU memory). To save A100 time, run B4 alone in an A100 session with `CONDITIONS = ['B4']` while a T4 session trains the rest.


In [ ]:
# ============== SETTINGS: the only cell you edit ==============
REPO_REF    = "main"     # branch or release tag (e.g. "v1.0.0") of the repository
STORAGE     = "drive"    # "drive": Google Drive (persists) | "local": /content (lost at disconnect)
DATA_ROOT   = None       # None: Colab secret NIDS_DATA_DIR, else MyDrive/NIDS_datasets
OUTPUT_ROOT = None       # None: Colab secret NIDS_OUTPUT_DIR, else MyDrive/NIDS_outputs
# Experiment
CONDITIONS = ['M', 'B2', 'B3', 'B4', 'A1', 'A2', 'A3']   # any subset of M, B2, B3, B4, A1, A2, A3 (campaign order is kept)
SEEDS      = [0, 1, 2, 3, 4]   # any list, e.g. [5, 6]; finished seeds are skipped
EXECUTE    = False   # False: print the plan only. True: train.
RUNS_PER_SESSION = None   # units per execution; None = until done or disconnected
BUILD_TABLES = True  # after training: cross-seed summary and paper tables once everything is complete
RUN_ANALYSES = True  # after that: per-class bias tuning for every condition (supplementary)
# ===============================================================

In [ ]:
# Setup: clone the repository, install it, mount storage and resolve the roots.
import os, subprocess, sys

REPO_URL, TARGET = "https://github.com/selimsidan/dmoe_nids.git", "/content/dmoe_nids"

def _git(*args, cwd=None, env=None):
    subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True,
                   env={**os.environ, **(env or {})})

def _fetch(env=None):
    if os.path.isdir(os.path.join(TARGET, ".git")):
        _git("fetch", "--quiet", "--tags", "origin", cwd=TARGET, env=env)
    else:
        _git("clone", "--quiet", REPO_URL, TARGET, env=env)

try:
    _fetch()
except subprocess.CalledProcessError:
    # Only for a private repository or fork: retry with the Colab secret GITHUB_TOKEN (never printed).
    import base64
    from google.colab import userdata
    _token = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    _fetch({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {_token}"})
    del _token
_git("checkout", "--quiet", REPO_REF, cwd=TARGET)
if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=TARGET).returncode == 0:
    # Fast-forward to the commits fetched above (no second download, so no token needed).
    _git("merge", "--quiet", "--ff-only", f"origin/{REPO_REF}", cwd=TARGET)

# Installs the package and any missing dependencies; Colab's own torch build is kept.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", TARGET], check=True)
sys.path.insert(0, os.path.join(TARGET, "src"))
os.environ["PYTHONPATH"] = os.path.join(TARGET, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(TARGET)

import dmoe.colab
ROOTS = dmoe.colab.setup(STORAGE, DATA_ROOT, OUTPUT_ROOT)

In [ ]:
# Preflight: hardware, data files, and the plan (completed work is skipped).
import os
from dmoe.colab import gpu_name
from dmoe.data.verify import verify_data
from dmoe.training.campaign import load_campaign, plan_units, print_plan

CAMPAIGN = load_campaign()
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"GPU: {gpu_name()}   RAM: {ram_gib:.0f} GiB   CPU cores: {os.cpu_count()}")
if gpu_name() is None:
    print("WARNING: no GPU. Training is impractically slow on CPU; switch the runtime type.")
if "B4" in CONDITIONS and not any(name in (gpu_name() or "") for name in ("A100", "H100")):
    print("WARNING: B4 (XGBoost on all 46.9M training rows) needs a GPU with about 40 GB of memory; "
          "choose an A100 runtime (Runtime > Change runtime type).")
checks = verify_data(ROOTS['data'].path, count=False, hash_files=False)  # existence and header
assert all(c.ok for c in checks), 'fix the data problems above (docs/DATA.md)'
print_plan(plan_units(CAMPAIGN, CONDITIONS, SEEDS, ROOTS['output'].path))

In [ ]:
# Train. Output streams here and into runs/<condition>/seed<k>/log.txt (B4 tuning: runs/B4/tuning/log.txt).
from dmoe.training.campaign import run_campaign

units = run_campaign(CAMPAIGN, CONDITIONS, SEEDS, execute=EXECUTE,
                     max_units=RUNS_PER_SESSION, output_root=ROOTS['output'].path)

In [ ]:
# Per-run results (test split) of the selected conditions and seeds.
import os, pandas as pd
from IPython.display import display

rows = []
for condition in CONDITIONS:
    for seed in SEEDS:
        path = os.path.join(ROOTS['output'].path, 'runs', condition, f'seed{seed}', 'eval', 'test', 'Overall_Metrics.csv')
        if os.path.isfile(path):
            frame = pd.read_csv(path)
            rows.append(frame[frame['origin'] == 'ALL'])
if rows:
    table = pd.concat(rows)[['condition', 'seed', 'macro_f1', 'accuracy', 'balanced_accuracy', 'weighted_f1']]
    display(table.reset_index(drop=True))
else:
    print('No completed runs yet for these settings.')

In [ ]:
# Cross-seed summary and paper tables, once every selected (condition, seed) is complete.
import os, pandas as pd
from IPython.display import Image, display
from dmoe.evaluation.aggregate import aggregate
from dmoe.evaluation.paper_tables import make_paper_tables

ALL_CONDITIONS = CAMPAIGN.condition_ids
status = plan_units(CAMPAIGN, ALL_CONDITIONS, SEEDS, ROOTS['output'].path)
todo = [u.describe() for u in status if u.status != 'complete']
if todo:
    print(f'{len(todo)} unit(s) not complete yet; tables are built when all are done:')
    for line in todo:
        print('  ', line)
elif BUILD_TABLES:
    OUT = ROOTS['output'].path
    aggregate(OUT, conditions=ALL_CONDITIONS, seeds=SEEDS, resamples=CAMPAIGN.bootstrap_resamples,
              aggregation_seed=CAMPAIGN.aggregation_seed)
    paper = make_paper_tables(OUT)
    for table in ('Table_Main', 'Table_Ablations'):
        print(table); display(pd.read_csv(os.path.join(paper, table + '.csv')))
    figure = os.path.join(paper, 'Figure_Confusion_M.png')
    if os.path.isfile(figure):
        display(Image(figure, width=900))

In [ ]:
# Supplementary analysis once every condition is complete (resumes if interrupted):
# per-class offsets tuned on validation for macro-F1, for every condition (docs/METHOD.md).
import os, subprocess, sys, pandas as pd
from IPython.display import display

def run_script(name, *args):
    """Run a repository script and stream its output into this cell."""
    command = [sys.executable, '-u', os.path.join(TARGET, 'scripts', name),
               '--data-root', ROOTS['data'].path, '--output-root', ROOTS['output'].path,
               '--scratch-root', ROOTS['scratch'].path, *args]
    with subprocess.Popen(command, cwd=TARGET, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
    if process.returncode:
        raise RuntimeError(f'{name} failed with exit code {process.returncode} (see the output above)')

if RUN_ANALYSES:
    status = plan_units(CAMPAIGN, CAMPAIGN.condition_ids, SEEDS, ROOTS['output'].path)
    if all(u.status == 'complete' for u in status):
        run_script('bias_tuning.py', '--seeds', *[str(s) for s in SEEDS])
        paired = pd.read_csv(os.path.join(ROOTS['output'].path, 'summary', 'Bias_Tuning_Paired_vs_M.csv'))
        display(paired[(paired.origin == 'ALL') & (paired.metric == 'macro_f1')])
    else:
        print('The analysis runs once every unit of every condition is complete.')